# H8: Clinical Covariates × Spleen Morphology

**Aim:** Test whether HANDEL clinical variables (age, BMI, C-peptide, HbA1c) predict
spleen architecture independently of rs3184504 genotype, and whether genotype effects
survive as partial correlations.

**Context:** All subjects are Type 1 Diabetes donors from the HANDEL cohort. Clinical
variables may confound or modify the genotype–morphology relationship. SH2B3/LNK
modulates JAK/STAT signaling in immune and vascular cells; clinical T1D markers
(C-peptide, HbA1c) may independently affect spleen architecture.

**Caveats:**
- Small sample size (n=10–15 depending on platform)
- Missing data for some clinical variables (especially CODEX samples)
- CODEX vs Phenocycler platform confound (p=0.019 for Follicle_norm_density)
- All analyses run dual cohort: All samples and Phenocycler-only

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact, spearmanr

from data_utils import (
    load_all_data, build_feature_matrix, load_clinical, compute_density,
    partial_spearman, platform_diagnostic, run_kruskal,
    GENO_ORDER, GENO_PALETTE, MAIN_REGIONS, CODEX_SAMPLES,
    CONTINUOUS_CLINICAL, CLINICAL_COLS,
    setup_style, save_figure, save_table,
)

setup_style()

# Load and merge
df = load_all_data()
feat = build_feature_matrix(df)
clin = load_clinical()

# Merge clinical onto feature matrix (inner join on project samples)
merged = feat.reset_index().merge(clin.drop(columns=['Genotype', 'Platform']),
                                   on='Sample', how='inner')
merged_pc = merged[merged['Platform'] == 'Phenocycler'].copy()

# Identify morphological feature columns
morph_cols = [c for c in feat.columns if c not in ('Genotype', 'Platform')]
# Top morphological features for scatter plots
top_morph = ['Follicle_density', 'Follicle_norm_density', 'RedPulp_density',
             'Follicle_count', 'Follicle_fraction', 'WP_fraction',
             'Vessel_median_area', 'Vessel_median_circularity']
top_morph = [c for c in top_morph if c in morph_cols]

print(f'Merged samples: {len(merged)} (All), {len(merged_pc)} (PC-only)')
print(f'Morphological features: {len(morph_cols)}')
print(f'Clinical columns with data: {clin[CONTINUOUS_CLINICAL].notna().sum().to_dict()}')

## Section 1: Genotype Group Balance on Demographics

In [ ]:
# Demographics summary table by genotype
rows = []
for geno in GENO_ORDER:
    sub = merged[merged['Genotype'] == geno]
    row = {'Genotype': geno, 'N': len(sub)}
    for col in CONTINUOUS_CLINICAL:
        vals = sub[col].dropna()
        row[f'{col}_mean'] = vals.mean()
        row[f'{col}_SD'] = vals.std()
        row[f'{col}_median'] = vals.median()
        row[f'{col}_n'] = len(vals)
    # Gender counts
    gc = sub['Gender'].value_counts()
    row['Female'] = gc.get('Female', 0)
    row['Male'] = gc.get('Male', 0)
    # Ethnicity counts
    ec = sub['Ethnicity'].value_counts()
    for eth in merged['Ethnicity'].dropna().unique():
        row[f'Eth_{eth}'] = ec.get(eth, 0)
    rows.append(row)

demo_df = pd.DataFrame(rows)

# KW tests for continuous variables
kw_rows = []
for col in CONTINUOUS_CLINICAL:
    h, p = run_kruskal(merged, col)
    kw_rows.append({'Variable': col, 'Test': 'Kruskal-Wallis', 'H': h, 'p': p})

# Fisher exact for Gender (collapse to 2x3 → pairwise 2x2)
gender_table = pd.crosstab(merged['Genotype'], merged['Gender'])
if gender_table.shape == (3, 2):
    # Use chi2 approximation for 3x2
    from scipy.stats import chi2_contingency
    chi2, p_gender, _, _ = chi2_contingency(gender_table)
    kw_rows.append({'Variable': 'Gender', 'Test': 'Chi-squared', 'H': chi2, 'p': p_gender})

balance_stats = pd.DataFrame(kw_rows)
save_table(demo_df, 'H8_demographics_balance')
display(demo_df)
display(balance_stats)

### Figure 1: Clinical Variables by Genotype

In [ ]:
avail_clinical = [c for c in CONTINUOUS_CLINICAL if merged[c].notna().sum() >= 3]
ncols = min(len(avail_clinical), 4)
nrows = (len(avail_clinical) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)

for i, col in enumerate(avail_clinical):
    ax = axes[i // ncols, i % ncols]
    sns.boxplot(data=merged, x='Genotype', y=col, order=GENO_ORDER,
                palette=GENO_PALETTE, ax=ax, linewidth=0.8, fliersize=0)
    sns.stripplot(data=merged, x='Genotype', y=col, order=GENO_ORDER,
                  palette=GENO_PALETTE, ax=ax, size=7, jitter=0.15, edgecolor='black',
                  linewidth=0.5)
    ax.set_title(col)
    ax.set_xlabel('')

# Hide unused axes
for j in range(len(avail_clinical), nrows * ncols):
    axes[j // ncols, j % ncols].set_visible(False)

fig.suptitle('Clinical Variables by rs3184504 Genotype', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, 'H8_clinical_by_genotype')
plt.show()

## Section 2: Clinical × Morphological Correlation Matrix

In [ ]:
def spearman_matrix(data, x_cols, y_cols):
    """Compute Spearman correlation matrix between two sets of columns."""
    rho_mat = pd.DataFrame(index=x_cols, columns=y_cols, dtype=float)
    p_mat = pd.DataFrame(index=x_cols, columns=y_cols, dtype=float)
    for xc in x_cols:
        for yc in y_cols:
            valid = data[[xc, yc]].dropna()
            if len(valid) < 4:
                rho_mat.loc[xc, yc] = np.nan
                p_mat.loc[xc, yc] = np.nan
            else:
                r, p = spearmanr(valid[xc], valid[yc])
                rho_mat.loc[xc, yc] = r
                p_mat.loc[xc, yc] = p
    return rho_mat.astype(float), p_mat.astype(float)

clin_avail = [c for c in CONTINUOUS_CLINICAL if merged[c].notna().sum() >= 4]
morph_sub = top_morph[:8]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, data_sub, title in [(axes[0], merged, f'All (n={len(merged)})'),
                             (axes[1], merged_pc, f'Phenocycler-only (n={len(merged_pc)})')]:
    rho, pvals = spearman_matrix(data_sub, clin_avail, morph_sub)
    # Annotate with significance stars
    annot = rho.copy().astype(str)
    for xc in rho.index:
        for yc in rho.columns:
            r_val = rho.loc[xc, yc]
            p_val = pvals.loc[xc, yc]
            star = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
            annot.loc[xc, yc] = f'{r_val:.2f}{star}'
    sns.heatmap(rho, annot=annot, fmt='', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                ax=ax, cbar_kws={'shrink': 0.8})
    ax.set_title(title)

fig.suptitle('Spearman Correlation: Clinical \u00d7 Morphological Features', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, 'H8_correlation_matrix')
plt.show()

### Figure 3: Top Clinical–Morphological Scatter Plots

In [ ]:
# Pick top 6 clinical-morphological pairs by |rho| from All cohort
rho_all, p_all = spearman_matrix(merged, clin_avail, morph_sub)
pairs = []
for xc in rho_all.index:
    for yc in rho_all.columns:
        r = rho_all.loc[xc, yc]
        if not np.isnan(r):
            pairs.append((xc, yc, abs(r), p_all.loc[xc, yc]))
pairs.sort(key=lambda x: -x[2])
top6 = pairs[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
markers = {'Phenocycler': 'o', 'CODEX': 's'}

for idx, (xc, yc, rval, pval) in enumerate(top6):
    ax = axes[idx // 3, idx % 3]
    for plat, marker in markers.items():
        sub = merged[merged['Platform'] == plat]
        for geno in GENO_ORDER:
            gs = sub[sub['Genotype'] == geno]
            ax.scatter(gs[xc], gs[yc], c=[GENO_PALETTE[geno]], marker=marker,
                       s=60, edgecolors='black', linewidths=0.5,
                       label=f'{geno} ({plat})' if idx == 0 else '')
    ax.set_xlabel(xc)
    ax.set_ylabel(yc)
    ax.set_title(f'\u03c1={rval:.2f}, p={pval:.3f}')

# Shared legend
handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=3,
               bbox_to_anchor=(0.5, 1.05), fontsize=9)

fig.suptitle('Top Clinical\u2013Morphological Correlations', fontsize=14, y=1.08)
fig.tight_layout()
save_figure(fig, 'H8_clinical_scatter')
plt.show()

## Section 3: Age as Confound

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Age vs Follicle density
ax = axes[0]
for geno in GENO_ORDER:
    sub = merged[merged['Genotype'] == geno]
    ax.scatter(sub['Age (yrs)'], sub['Follicle_density'],
               c=[GENO_PALETTE[geno]], label=geno, s=60, edgecolors='black', linewidths=0.5)
r_age, p_age = spearmanr(merged['Age (yrs)'].dropna(),
                          merged.loc[merged['Age (yrs)'].notna(), 'Follicle_density'])
ax.set_xlabel('Age (yrs)')
ax.set_ylabel('Follicle Density (vessels/mm\u00b2)')
ax.set_title(f'Age vs Follicle Density (\u03c1={r_age:.2f}, p={p_age:.3f})')
ax.legend()

# Partial correlation: genotype dosage vs Follicle density, controlling for age
ax = axes[1]
merged['Dosage'] = merged['Genotype'].astype(str).map({'C/C': 0, 'C/T': 1, 'T/T': 2})

rho_raw, p_raw = spearmanr(merged['Dosage'], merged['Follicle_density'])
rho_partial, p_partial = partial_spearman(merged, 'Dosage', 'Follicle_density', ['Age (yrs)'])

bars = ax.bar(['Raw Spearman', 'Partial (age-adjusted)'],
              [rho_raw, rho_partial], color=['steelblue', 'coral'],
              edgecolor='black', linewidth=0.8)
ax.set_ylabel('Spearman \u03c1')
ax.set_title('Genotype Dosage vs Follicle Density')
for bar, p in zip(bars, [p_raw, p_partial]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'p={p:.3f}', ha='center', fontsize=10)
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_ylim(-0.6, 0.6)

fig.tight_layout()
save_figure(fig, 'H8_age_morphology')
plt.show()

## Section 4: Partial Correlations — Genotype Dosage Controlling for Clinical Covariates

In [ ]:
# Identify significant clinical covariates (p < 0.1 with any morph feature)
sig_covariates = []
for cvar in clin_avail:
    for mvar in morph_sub:
        valid = merged[[cvar, mvar]].dropna()
        if len(valid) >= 4:
            _, p = spearmanr(valid[cvar], valid[mvar])
            if p < 0.1:
                sig_covariates.append(cvar)
                break

if not sig_covariates:
    sig_covariates = clin_avail[:2]  # fallback: use age + first available

print(f'Covariates to control for: {sig_covariates}')

merged['Dosage'] = merged['Genotype'].astype(str).map({'C/C': 0, 'C/T': 1, 'T/T': 2})

partial_rows = []
for mvar in morph_cols:
    # Raw
    valid = merged[['Dosage', mvar]].dropna()
    if len(valid) >= 4:
        rho_raw, p_raw = spearmanr(valid['Dosage'], valid[mvar])
    else:
        rho_raw, p_raw = np.nan, np.nan
    # Partial
    rho_part, p_part = partial_spearman(merged, 'Dosage', mvar, sig_covariates)
    partial_rows.append({
        'Feature': mvar,
        'rho_raw': rho_raw, 'p_raw': p_raw,
        'rho_partial': rho_part, 'p_partial': p_part,
        'Covariates': ', '.join(sig_covariates),
    })

partial_df = pd.DataFrame(partial_rows)
save_table(partial_df, 'H8_partial_correlations')
display(partial_df.sort_values('p_partial'))

## Section 5: T1D Markers — C-peptide and HbA1c vs Morphology

In [ ]:
t1d_markers = ['C-pep (ng/ml)', 'HbA1c']
t1d_avail = [m for m in t1d_markers if merged[m].notna().sum() >= 4]
morph_endpoints = top_morph[:4]

if t1d_avail:
    fig, axes = plt.subplots(len(t1d_avail), len(morph_endpoints),
                             figsize=(4 * len(morph_endpoints), 4 * len(t1d_avail)),
                             squeeze=False)
    for i, t1d_var in enumerate(t1d_avail):
        for j, mvar in enumerate(morph_endpoints):
            ax = axes[i, j]
            valid = merged[[t1d_var, mvar, 'Genotype', 'Platform']].dropna()
            for geno in GENO_ORDER:
                gs = valid[valid['Genotype'] == geno]
                ax.scatter(gs[t1d_var], gs[mvar], c=[GENO_PALETTE[geno]],
                           label=geno, s=60, edgecolors='black', linewidths=0.5)
            if len(valid) >= 4:
                r, p = spearmanr(valid[t1d_var], valid[mvar])
                ax.set_title(f'\u03c1={r:.2f}, p={p:.3f}', fontsize=10)
            ax.set_xlabel(t1d_var)
            ax.set_ylabel(mvar)
            if i == 0 and j == 0:
                ax.legend(fontsize=8)

    fig.suptitle('T1D Markers vs Spleen Morphology', fontsize=14, y=1.02)
    fig.tight_layout()
    save_figure(fig, 'H8_t1d_markers')
    plt.show()
else:
    print('Insufficient T1D marker data for plotting.')

## Summary Tables

In [ ]:
# Full clinical-morphological correlation table
corr_rows = []
for cohort_name, data_sub in [('All', merged), ('Phenocycler', merged_pc)]:
    for cvar in clin_avail:
        for mvar in morph_cols:
            valid = data_sub[[cvar, mvar]].dropna()
            if len(valid) >= 4:
                r, p = spearmanr(valid[cvar], valid[mvar])
            else:
                r, p = np.nan, np.nan
            corr_rows.append({
                'Cohort': cohort_name, 'Clinical': cvar, 'Morphological': mvar,
                'rho': r, 'p': p, 'n': len(valid)
            })

corr_df = pd.DataFrame(corr_rows)
save_table(corr_df, 'H8_clinical_correlations')

# Statistical tests summary
stats_rows = []
# Demographics balance
for _, row in balance_stats.iterrows():
    stats_rows.append({
        'Section': 'Demographics Balance', 'Test': row['Test'],
        'Variable': row['Variable'], 'Statistic': row['H'], 'p': row['p']
    })
# Top partial correlations
for _, row in partial_df.iterrows():
    stats_rows.append({
        'Section': 'Partial Correlation (Dosage)', 'Test': 'Partial Spearman',
        'Variable': row['Feature'], 'Statistic': row['rho_partial'], 'p': row['p_partial']
    })

stats_df = pd.DataFrame(stats_rows)
save_table(stats_df, 'H8_statistical_tests')
print(f'\nSaved {len(corr_df)} correlations and {len(stats_df)} statistical tests.')

In [ ]:
# Key findings summary
print('=' * 60)
print('H8: Clinical Covariates \u00d7 Spleen Morphology \u2014 Key Findings')
print('=' * 60)

print(f'\nSamples: {len(merged)} (All), {len(merged_pc)} (Phenocycler-only)')

# Demographics balance
sig_demo = balance_stats[balance_stats['p'] < 0.05]
if len(sig_demo) > 0:
    print(f'\nSignificant demographic imbalances (p<0.05):')
    for _, r in sig_demo.iterrows():
        print(f'  {r["Variable"]}: {r["Test"]} p={r["p"]:.3f}')
else:
    print('\nNo significant demographic imbalances across genotype groups.')

# Top correlations
sig_corr = corr_df[(corr_df['Cohort'] == 'All') & (corr_df['p'] < 0.05)]
if len(sig_corr) > 0:
    print(f'\nSignificant clinical-morphological correlations (All, p<0.05): {len(sig_corr)}')
    for _, r in sig_corr.nsmallest(5, 'p').iterrows():
        print(f'  {r["Clinical"]} vs {r["Morphological"]}: rho={r["rho"]:.3f}, p={r["p"]:.3f}')

# Partial correlations
sig_partial = partial_df[partial_df['p_partial'] < 0.1]
if len(sig_partial) > 0:
    print(f'\nGenotype dosage effects surviving partial correlation (p<0.1):')
    for _, r in sig_partial.iterrows():
        print(f'  {r["Feature"]}: raw rho={r["rho_raw"]:.3f}, partial rho={r["rho_partial"]:.3f} (p={r["p_partial"]:.3f})')
else:
    print('\nNo genotype dosage effects reached p<0.1 after controlling for clinical covariates.')